# vLLM Automatic Prefix Caching Benchmark

This notebook compares vLLM with Automatic Prefix Caching disabled and enabled under matched serving workloads. It measures time to first token, end-to-end latency, request throughput, prompt-token throughput, concurrency, shared-prefix length, output length, and cache pressure.

In [ ]:
import os
import sys
import subprocess

# Reproducible current vLLM line. Run this before importing torch/vllm.
PACKAGES = [
    "vllm==0.24.0",
    "bitsandbytes>=0.49.2,<1",
    "transformers>=4.57,<5",
    "aiohttp>=3.10,<4",
        "requests>=2.32,<3",
    "pandas>=2.2,<3",
    "numpy>=1.26,<3",
    "matplotlib>=3.8,<4",
    "prometheus-client>=0.21,<1",
    "psutil>=5.9,<8",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", *PACKAGES
])
print("Installed production benchmark dependencies.")

In [ ]:
import asyncio
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import signal
import socket
import statistics
import subprocess
import sys
import time
import traceback
import zipfile
from collections import defaultdict
from pathlib import Path

import aiohttp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import transformers
from transformers import AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running.")

ROOT = Path("/kaggle/working/mosaic_vllm_apc")
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
LOGS = ROOT / "logs"
for directory in (RESULTS, FIGURES, LOGS):
    directory.mkdir(parents=True, exist_ok=True)

GPU_NAME = torch.cuda.get_device_name(0)
GPU_BYTES = torch.cuda.get_device_properties(0).total_memory
VLLM_EXE = shutil.which("vllm")
if not VLLM_EXE:
    raise RuntimeError("vllm CLI was not installed.")

print("GPU:", GPU_NAME, f"{GPU_BYTES / 2**30:.1f} GiB")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("vllm executable:", VLLM_EXE)

## Execution profiles

`paper_t4` runs the primary small-model conditions. Use `paper_small` when a 7B model does not fit on a T4, and use `paper_7b` on a larger GPU.

In [ ]:
def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token configured for gated models.")
else:
    print("No HF_TOKEN configured; public default models still work.")

In [ ]:
PROFILE = "paper_t4"

PROFILES = {
    "smoke": {
        "models": "small",
        "contexts": [512, 1024],
        "workloads": ["agent"],
        "concurrencies": [1, 8],
        "rounds": 1,
        "requests_per_batch": 8,
        "run_trace": False,
        "run_output_sensitivity": False,
        "run_eviction": False,
    },
    "paper_small": {
        "models": "small",
        "contexts": [512, 1024, 2048],
        "workloads": ["agent", "rag"],
        "concurrencies": [1, 8, 16],
        "rounds": 2,
        "requests_per_batch": 16,
        "run_trace": True,
        "run_output_sensitivity": True,
        "run_eviction": False,
    },
    "paper_t4": {
        "models": "t4",
        "contexts": [512, 1024, 2048],
        "workloads": ["agent", "rag"],
        "concurrencies": [1, 8, 16],
        "rounds": 2,
        "requests_per_batch": 16,
        "run_trace": True,
        "run_output_sensitivity": True,
        "run_eviction": False,
    },
    "paper_7b": {
        "models": "7b",
        "contexts": [512, 1024, 2048, 4096],
        "workloads": ["agent", "rag", "repository"],
        "concurrencies": [1, 8, 16, 32],
        "rounds": 3,
        "requests_per_batch": 32,
        "run_trace": True,
        "run_output_sensitivity": True,
        "run_eviction": True,
    },
}

MODEL_SUITES = {
    "small": [
        {
            "family": "Llama-architecture",
            "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            "quantization": None,
        },
        {
            "family": "Qwen",
            "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
            "quantization": None,
        },
    ],
    "t4": [
        {
            "family": "Llama-architecture",
            "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            "quantization": None,
        },
        {
            "family": "Qwen",
            "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
            "quantization": None,
        },
        {
            "family": "Mistral",
            "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
            "quantization": "bitsandbytes",
        },
    ],
    "7b": [
        {
            "family": "Llama",
            "model_id": "meta-llama/Llama-3.1-8B-Instruct",
            "quantization": "bitsandbytes",
        },
        {
            "family": "Qwen",
            "model_id": "Qwen/Qwen2.5-7B-Instruct",
            "quantization": "bitsandbytes",
        },
        {
            "family": "Mistral",
            "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
            "quantization": "bitsandbytes",
        },
    ],
}

CFG = PROFILES[PROFILE]
MODELS = MODEL_SUITES[CFG["models"]]
SERVER_MAX_MODEL_LEN = 4608
SERVER_GPU_MEMORY_UTILIZATION = 0.90
SERVER_START_TIMEOUT_S = 900
SERVER_STOP_TIMEOUT_S = 60
PORT_BASE = 8100
RANDOM_SEED = 20260721

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(json.dumps(CFG, indent=2))
print([model["model_id"] for model in MODELS])

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful repository and retrieval analysis agent. "
    "Preserve exact symbol names, cite paths, never invent execution results, "
    "and clearly separate verified facts from hypotheses."
)

TOOL_SCHEMA = json.dumps({
    "tools": [
        {
            "name": "search_repository",
            "description": "Search source files.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "path": {"type": "string"},
                    "limit": {"type": "integer"},
                },
                "required": ["query"],
            },
        },
        {
            "name": "run_tests",
            "description": "Run a named test target.",
            "parameters": {
                "type": "object",
                "properties": {
                    "target": {"type": "string"},
                    "timeout": {"type": "integer"},
                },
                "required": ["target"],
            },
        },
    ]
}, indent=2)

RAG_DOCUMENTS = "\n\n".join([
    "Document A: Exact prefix reuse is safe only when the causal token prefix is identical.",
    "Document B: Cache identity includes the model revision, tokenizer, and exact token content.",
    "Document C: Request-local branches must not mutate the immutable shared prefix state.",
    "Document D: TTFT includes queueing, scheduling, cache lookup, prefill, and first-token decode.",
    "Document E: Prefix caching accelerates prefill but not long answer decoding.",
] * 160)

REPOSITORY = "\n\n".join([
    "src/runtime/cache.py defines immutable block hashes and request-local branches.",
    "src/runtime/planner.py recognizes exact reusable prefixes and estimates benefit.",
    "src/runtime/executor.py binds cached state and executes the private suffix.",
    "tests/test_cache.py checks token equality, model fingerprints, and branch isolation.",
    "benchmarks/serve.py records TTFT, E2E latency, throughput, hit rate, and eviction.",
] * 192)

WORKLOADS = {
    "agent": SYSTEM_PROMPT + "\n\nAVAILABLE TOOLS:\n" + TOOL_SCHEMA,
    "rag": SYSTEM_PROMPT + "\n\nRETRIEVED CONTEXT:\n" + RAG_DOCUMENTS,
    "repository": SYSTEM_PROMPT + "\n\nREPOSITORY SNAPSHOT:\n" + REPOSITORY,
}

SUFFIXES = [
    "\nUser: Explain the cache invalidation policy.\nAssistant:",
    "\nUser: Find a likely branch mutation bug.\nAssistant:",
    "\nUser: Propose exact-prefix correctness tests.\nAssistant:",
    "\nUser: Estimate the compile break-even point.\nAssistant:",
    "\nUser: Compare cold and warm request behavior.\nAssistant:",
    "\nUser: Identify claims requiring a full serving benchmark.\nAssistant:",
    "\nUser: Explain the memory and eviction tradeoff.\nAssistant:",
    "\nUser: Give one safe negative-control workload.\nAssistant:",
]

In [ ]:
def find_free_port(start=8100):
    for port in range(start, start + 200):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("127.0.0.1", port))
                return port
            except OSError:
                continue
    raise RuntimeError("No free local port found.")

def process_tree_kill(process):
    if process is None:
        return
    try:
        parent = psutil.Process(process.pid)
        children = parent.children(recursive=True)
        for child in children:
            child.terminate()
        parent.terminate()
        _, alive = psutil.wait_procs(
            children + [parent], timeout=SERVER_STOP_TIMEOUT_S
        )
        for proc in alive:
            proc.kill()
    except Exception:
        try:
            process.kill()
        except Exception:
            pass

def wait_for_server(base_url, process, timeout_s):
    import requests

    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        if process.poll() is not None:
            raise RuntimeError(
                f"vLLM server exited early with code {process.returncode}"
            )
        try:
            response = requests.get(
                f"{base_url}/v1/models", timeout=5
            )
            if response.status_code == 200:
                return response.json()
            last_error = RuntimeError(
                f"HTTP {response.status_code}: {response.text[:300]}"
            )
        except Exception as error:
            last_error = error
        time.sleep(3)
    raise TimeoutError(f"Server did not become ready: {last_error}")

def launch_server(spec, apc_enabled, port, kv_cache_memory=None):
    log_name = (
        spec["family"].replace("/", "_").replace(" ", "_")
        + ("_apc_on.log" if apc_enabled else "_apc_off.log")
    )
    log_path = LOGS / log_name
    log_handle = open(log_path, "w", encoding="utf-8")

    command = [
        VLLM_EXE,
        "serve",
        spec["model_id"],
        "--host", "127.0.0.1",
        "--port", str(port),
        "--served-model-name", "mosaic-bench",
        "--dtype", "half",
        "--max-model-len", str(SERVER_MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(SERVER_GPU_MEMORY_UTILIZATION),
        "--seed", str(RANDOM_SEED),
    ]

    if apc_enabled:
        command.append("--enable-prefix-caching")
    else:
        command.append("--no-enable-prefix-caching")

    if spec.get("quantization"):
        command.extend([
            "--quantization", spec["quantization"],
        ])

    if kv_cache_memory:
        command.extend([
            "--kv-cache-memory-bytes", str(kv_cache_memory)
        ])

    env = dict(os.environ)
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["VLLM_LOGGING_LEVEL"] = "INFO"

    print("Launching:", " ".join(command))
    process = subprocess.Popen(
        command,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=env,
        text=True,
    )
    process._mosaic_log_handle = log_handle
    return process, log_path

def stop_server(process):
    if process is None:
        return
    process_tree_kill(process)
    try:
        process._mosaic_log_handle.close()
    except Exception:
        pass
    time.sleep(5)

In [ ]:
def parse_prometheus(text):
    totals = defaultdict(float)
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        match = re.match(
            r"^([a-zA-Z_:][a-zA-Z0-9_:]*)(?:\{[^}]*\})?\s+"
            r"([-+0-9.eE]+)$",
            line,
        )
        if not match:
            continue
        name, value = match.groups()
        try:
            totals[name] += float(value)
        except ValueError:
            continue
    return dict(totals)

async def fetch_metrics(session, base_url):
    try:
        async with session.get(
            f"{base_url}/metrics",
            timeout=aiohttp.ClientTimeout(total=20),
        ) as response:
            text = await response.text()
            if response.status != 200:
                return {}
            return parse_prometheus(text)
    except Exception:
        return {}

def metric_delta(after, before, names):
    for name in names:
        if name in after or name in before:
            return after.get(name, 0.0) - before.get(name, 0.0)
    return float("nan")

PREFIX_QUERY_METRICS = [
    "vllm:prefix_cache_queries",
    "vllm:prefix_cache_queries_total",
]
PREFIX_HIT_METRICS = [
    "vllm:prefix_cache_hits",
    "vllm:prefix_cache_hits_total",
]
PROMPT_TOKEN_METRICS = [
    "vllm:prompt_tokens_total",
]
GENERATION_TOKEN_METRICS = [
    "vllm:generation_tokens_total",
]

In [ ]:
def exact_length_text(tokenizer, source, target_tokens, nonce):
    header = f"[MOSAIC-CONDITION:{nonce}]\n"
    base = header + source
    encoded = tokenizer.encode(base, add_special_tokens=False)

    if len(encoded) < target_tokens:
        filler = (
            "\nThe runtime must preserve exact causal state, request isolation, "
            "token identity, model revision, and deterministic cache keys."
        )
        while len(encoded) < target_tokens:
            base += filler
            encoded = tokenizer.encode(base, add_special_tokens=False)

    prefix_ids = encoded[:target_tokens]
    decoded = tokenizer.decode(
        prefix_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    # Record actual re-tokenized length. Exact equality across requests is
    # guaranteed because the same decoded string is reused.
    actual_ids = tokenizer.encode(decoded, add_special_tokens=False)
    return decoded, len(actual_ids)

def make_prompts(
    tokenizer,
    workload_name,
    target_prefix_tokens,
    request_count,
    nonce,
):
    prefix, actual_prefix_tokens = exact_length_text(
        tokenizer,
        WORKLOADS[workload_name],
        target_prefix_tokens,
        nonce,
    )
    prompts = []
    for index in range(request_count):
        suffix = SUFFIXES[index % len(SUFFIXES)]
        prompts.append(prefix + suffix + f"\nRequest-ID: {index}")
    prime_prompt = prefix + "\nUser: Prime the immutable prefix.\nAssistant:"
    return prime_prompt, prompts, actual_prefix_tokens

In [ ]:
async def request_one(
    session,
    base_url,
    prompt,
    max_tokens,
    request_id,
    timeout_s=600,
):
    payload = {
        "model": "mosaic-bench",
        "prompt": prompt,
        "max_tokens": int(max_tokens),
        "temperature": 0.0,
        "seed": RANDOM_SEED,
        "stream": True,
        "stream_options": {"include_usage": True},
        "ignore_eos": True,
    }

    start = time.perf_counter_ns()
    first_token_ns = None
    output_fragments = []
    usage = {}
    status = None
    error = None

    try:
        timeout = aiohttp.ClientTimeout(total=timeout_s)
        async with session.post(
            f"{base_url}/v1/completions",
            json=payload,
            timeout=timeout,
        ) as response:
            status = response.status
            if response.status != 200:
                body = await response.text()
                raise RuntimeError(
                    f"HTTP {response.status}: {body[:1000]}"
                )

            buffer = ""
            async for chunk in response.content.iter_any():
                buffer += chunk.decode("utf-8", errors="replace")
                while "\n\n" in buffer:
                    event, buffer = buffer.split("\n\n", 1)
                    for line in event.splitlines():
                        if not line.startswith("data:"):
                            continue
                        data = line[5:].strip()
                        if not data or data == "[DONE]":
                            continue
                        obj = json.loads(data)
                        if obj.get("usage"):
                            usage = obj["usage"]
                        choices = obj.get("choices") or []
                        if choices:
                            text = choices[0].get("text", "")
                            if text:
                                if first_token_ns is None:
                                    first_token_ns = time.perf_counter_ns()
                                output_fragments.append(text)

            # Some servers may omit a non-empty first fragment for special
            # tokens; then use completion time as a conservative TTFT.
            if first_token_ns is None:
                first_token_ns = time.perf_counter_ns()

    except Exception as exc:
        error = repr(exc)

    end = time.perf_counter_ns()
    ttft_ms = (
        (first_token_ns - start) / 1e6
        if first_token_ns is not None else float("nan")
    )
    e2e_ms = (end - start) / 1e6

    return {
        "request_id": request_id,
        "status": status,
        "error": error,
        "ttft_ms": ttft_ms,
        "e2e_ms": e2e_ms,
        "output_text": "".join(output_fragments),
        "prompt_tokens_api": usage.get("prompt_tokens"),
        "completion_tokens_api": usage.get("completion_tokens"),
        "total_tokens_api": usage.get("total_tokens"),
    }

async def run_batch(
    session,
    base_url,
    prompts,
    concurrency,
    max_tokens,
    request_id_prefix,
):
    semaphore = asyncio.Semaphore(concurrency)

    async def guarded(index, prompt):
        async with semaphore:
            return await request_one(
                session=session,
                base_url=base_url,
                prompt=prompt,
                max_tokens=max_tokens,
                request_id=f"{request_id_prefix}-{index}",
            )

    batch_start = time.perf_counter_ns()
    records = await asyncio.gather(*[
        guarded(index, prompt)
        for index, prompt in enumerate(prompts)
    ])
    batch_end = time.perf_counter_ns()
    return records, (batch_end - batch_start) / 1e6

In [ ]:
def percentile(values, q):
    array = np.asarray(values, dtype=np.float64)
    return float(np.quantile(array, q)) if len(array) else float("nan")

def bootstrap_speedup(off_values, on_values, samples=3000):
    off = np.asarray(off_values, dtype=np.float64)
    on = np.asarray(on_values, dtype=np.float64)
    if len(off) == 0 or len(on) == 0:
        return float("nan"), float("nan"), float("nan")

    rng = np.random.default_rng(RANDOM_SEED)
    estimates = []
    for _ in range(samples):
        off_sample = rng.choice(off, size=len(off), replace=True)
        on_sample = rng.choice(on, size=len(on), replace=True)
        denominator = np.median(on_sample)
        if denominator > 0:
            estimates.append(np.median(off_sample) / denominator)

    estimates = np.asarray(estimates)
    return (
        float(np.median(off) / np.median(on)),
        float(np.quantile(estimates, 0.025)),
        float(np.quantile(estimates, 0.975)),
    )

In [ ]:
async def benchmark_condition(
    session,
    tokenizer,
    base_url,
    spec,
    apc_enabled,
    workload,
    target_context,
    concurrency,
    round_index,
    request_count,
    max_tokens=1,
    phase="length_sweep",
):
    nonce = (
        f"{spec['family']}-{phase}-{workload}-"
        f"{target_context}-{concurrency}-{round_index}-{max_tokens}"
    )
    prime_prompt, prompts, actual_prefix_tokens = make_prompts(
        tokenizer=tokenizer,
        workload_name=workload,
        target_prefix_tokens=target_context,
        request_count=request_count,
        nonce=nonce,
    )

    metrics_before = await fetch_metrics(session, base_url)

    prime = await request_one(
        session=session,
        base_url=base_url,
        prompt=prime_prompt,
        max_tokens=1,
        request_id=f"{nonce}-prime",
    )

    metrics_after_prime = await fetch_metrics(session, base_url)

    records, batch_wall_ms = await run_batch(
        session=session,
        base_url=base_url,
        prompts=prompts,
        concurrency=concurrency,
        max_tokens=max_tokens,
        request_id_prefix=nonce,
    )

    metrics_after = await fetch_metrics(session, base_url)

    successful = [row for row in records if row["error"] is None]
    errors = [row for row in records if row["error"] is not None]

    ttfts = [row["ttft_ms"] for row in successful]
    e2es = [row["e2e_ms"] for row in successful]
    completion_tokens = sum(
        int(row["completion_tokens_api"] or max_tokens)
        for row in successful
    )
    prompt_tokens = sum(
        int(row["prompt_tokens_api"] or 0)
        for row in successful
    )

    prefix_queries = metric_delta(
        metrics_after, metrics_after_prime, PREFIX_QUERY_METRICS
    )
    prefix_hits = metric_delta(
        metrics_after, metrics_after_prime, PREFIX_HIT_METRICS
    )
    hit_rate = (
        prefix_hits / prefix_queries
        if np.isfinite(prefix_queries) and prefix_queries > 0
        else float("nan")
    )

    return {
        "family": spec["family"],
        "model_id": spec["model_id"],
        "quantization": spec.get("quantization") or "fp16",
        "apc_enabled": apc_enabled,
        "phase": phase,
        "workload": workload,
        "target_prefix_tokens": target_context,
        "actual_prefix_tokens": actual_prefix_tokens,
        "concurrency": concurrency,
        "round": round_index,
        "requests": request_count,
        "max_tokens": max_tokens,
        "successful_requests": len(successful),
        "failed_requests": len(errors),
        "prime_ttft_ms": prime["ttft_ms"],
        "prime_e2e_ms": prime["e2e_ms"],
        "median_ttft_ms": percentile(ttfts, 0.5),
        "p50_ttft_ms": percentile(ttfts, 0.5),
        "p90_ttft_ms": percentile(ttfts, 0.9),
        "p99_ttft_ms": percentile(ttfts, 0.99),
        "median_e2e_ms": percentile(e2es, 0.5),
        "p90_e2e_ms": percentile(e2es, 0.9),
        "batch_wall_ms": batch_wall_ms,
        "requests_per_s": (
            len(successful) / (batch_wall_ms / 1000)
            if batch_wall_ms > 0 else float("nan")
        ),
        "prompt_tokens_per_s": (
            prompt_tokens / (batch_wall_ms / 1000)
            if batch_wall_ms > 0 else float("nan")
        ),
        "completion_tokens_per_s": (
            completion_tokens / (batch_wall_ms / 1000)
            if batch_wall_ms > 0 else float("nan")
        ),
        "prefix_cache_queries_delta": prefix_queries,
        "prefix_cache_hits_delta": prefix_hits,
        "prefix_cache_hit_rate": hit_rate,
        "prompt_tokens_metric_delta": metric_delta(
            metrics_after, metrics_after_prime, PROMPT_TOKEN_METRICS
        ),
        "generation_tokens_metric_delta": metric_delta(
            metrics_after, metrics_after_prime, GENERATION_TOKEN_METRICS
        ),
        "prime_prefix_queries_delta": metric_delta(
            metrics_after_prime, metrics_before, PREFIX_QUERY_METRICS
        ),
        "prime_prefix_hits_delta": metric_delta(
            metrics_after_prime, metrics_before, PREFIX_HIT_METRICS
        ),
        "amortized_total_ms": prime["e2e_ms"] + batch_wall_ms,
        "errors_json": json.dumps(errors),
        "request_records_json": json.dumps(records),
    }

In [ ]:
async def run_length_sweep(
    tokenizer,
    base_url,
    spec,
    apc_enabled,
):
    rows = []
    connector = aiohttp.TCPConnector(limit=128)
    async with aiohttp.ClientSession(connector=connector) as session:
        for workload in CFG["workloads"]:
            for context in CFG["contexts"]:
                for concurrency in CFG["concurrencies"]:
                    for round_index in range(CFG["rounds"]):
                        row = await benchmark_condition(
                            session=session,
                            tokenizer=tokenizer,
                            base_url=base_url,
                            spec=spec,
                            apc_enabled=apc_enabled,
                            workload=workload,
                            target_context=context,
                            concurrency=concurrency,
                            round_index=round_index,
                            request_count=CFG["requests_per_batch"],
                            max_tokens=1,
                            phase="length_sweep",
                        )
                        rows.append(row)
                        print(
                            spec["family"],
                            "APC", apc_enabled,
                            workload,
                            context,
                            "C", concurrency,
                            f"TTFT={row['median_ttft_ms']:.1f}ms",
                            f"RPS={row['requests_per_s']:.2f}",
                            f"hit={row['prefix_cache_hit_rate']:.3f}",
                        )
    return rows

In [ ]:
async def run_output_sensitivity(
    tokenizer,
    base_url,
    spec,
    apc_enabled,
):
    if not CFG["run_output_sensitivity"]:
        return []

    rows = []
    connector = aiohttp.TCPConnector(limit=64)
    async with aiohttp.ClientSession(connector=connector) as session:
        for max_tokens in (1, 16, 64):
            row = await benchmark_condition(
                session=session,
                tokenizer=tokenizer,
                base_url=base_url,
                spec=spec,
                apc_enabled=apc_enabled,
                workload="rag",
                target_context=max(CFG["contexts"]),
                concurrency=8,
                round_index=max_tokens,
                request_count=16,
                max_tokens=max_tokens,
                phase="output_sensitivity",
            )
            rows.append(row)
            print(
                spec["family"],
                "output", max_tokens,
                "APC", apc_enabled,
                f"TTFT={row['median_ttft_ms']:.1f}ms",
                f"E2E={row['median_e2e_ms']:.1f}ms",
            )
    return rows

In [ ]:
async def run_zipf_trace(
    tokenizer,
    base_url,
    spec,
    apc_enabled,
    trace_requests=96,
    prefix_count=12,
    concurrency=8,
):
    if not CFG["run_trace"]:
        return [], {}

    rng = np.random.default_rng(RANDOM_SEED)
    ranks = np.arange(1, prefix_count + 1)
    probabilities = 1.0 / np.power(ranks, 1.2)
    probabilities /= probabilities.sum()

    prefixes = []
    prefix_lengths = []
    for prefix_index in range(prefix_count):
        prefix, length = exact_length_text(
            tokenizer,
            WORKLOADS["rag"],
            1536,
            nonce=(
                f"trace-{spec['family']}-"
                f"{prefix_index}"
            ),
        )
        prefixes.append(prefix)
        prefix_lengths.append(length)

    selected = rng.choice(
        prefix_count,
        size=trace_requests,
        replace=True,
        p=probabilities,
    )
    prompts = [
        prefixes[int(prefix_id)]
        + SUFFIXES[index % len(SUFFIXES)]
        + f"\nTrace request: {index}"
        for index, prefix_id in enumerate(selected)
    ]

    connector = aiohttp.TCPConnector(limit=128)
    async with aiohttp.ClientSession(connector=connector) as session:
        before = await fetch_metrics(session, base_url)
        records, batch_ms = await run_batch(
            session=session,
            base_url=base_url,
            prompts=prompts,
            concurrency=concurrency,
            max_tokens=1,
            request_id_prefix=(
                f"trace-{spec['family']}-{apc_enabled}"
            ),
        )
        after = await fetch_metrics(session, base_url)

    successful = [row for row in records if row["error"] is None]
    queries = metric_delta(after, before, PREFIX_QUERY_METRICS)
    hits = metric_delta(after, before, PREFIX_HIT_METRICS)

    summary = {
        "family": spec["family"],
        "model_id": spec["model_id"],
        "apc_enabled": apc_enabled,
        "phase": "zipf_trace",
        "trace_requests": trace_requests,
        "unique_prefixes": prefix_count,
        "concurrency": concurrency,
        "median_prefix_tokens": float(np.median(prefix_lengths)),
        "successful_requests": len(successful),
        "batch_wall_ms": batch_ms,
        "requests_per_s": (
            len(successful) / (batch_ms / 1000)
            if batch_ms > 0 else float("nan")
        ),
        "median_ttft_ms": percentile(
            [row["ttft_ms"] for row in successful], 0.5
        ),
        "p90_ttft_ms": percentile(
            [row["ttft_ms"] for row in successful], 0.9
        ),
        "median_e2e_ms": percentile(
            [row["e2e_ms"] for row in successful], 0.5
        ),
        "prefix_cache_queries_delta": queries,
        "prefix_cache_hits_delta": hits,
        "prefix_cache_hit_rate": (
            hits / queries
            if np.isfinite(queries) and queries > 0
            else float("nan")
        ),
        "zipf_alpha": 1.2,
        "prefix_id_histogram": json.dumps(
            np.bincount(selected, minlength=prefix_count).tolist()
        ),
    }
    return records, summary

## Optional eviction probe

The eviction probe starts a server with restricted KV-cache capacity and checks whether an anchor prefix remains cached after additional traffic.

In [ ]:
async def run_eviction_probe(
    tokenizer,
    base_url,
    spec,
    apc_enabled,
    unique_prefixes=96,
    prefix_tokens=1024,
):
    if not CFG["run_eviction"] or not apc_enabled:
        return {}

    anchor, _, _ = make_prompts(
        tokenizer,
        "rag",
        prefix_tokens,
        1,
        nonce=f"eviction-anchor-{spec['family']}",
    )

    connector = aiohttp.TCPConnector(limit=32)
    async with aiohttp.ClientSession(connector=connector) as session:
        anchor_cold = await request_one(
            session, base_url, anchor, 1, "anchor-cold"
        )
        anchor_warm = await request_one(
            session, base_url, anchor, 1, "anchor-warm"
        )

        pressure_prompts = []
        for index in range(unique_prefixes):
            prefix, _ = exact_length_text(
                tokenizer,
                WORKLOADS["repository"],
                prefix_tokens,
                nonce=f"evict-{spec['family']}-{index}",
            )
            pressure_prompts.append(
                prefix + f"\nUser: Pressure request {index}.\nAssistant:"
            )

        _, pressure_ms = await run_batch(
            session,
            base_url,
            pressure_prompts,
            concurrency=8,
            max_tokens=1,
            request_id_prefix="eviction-pressure",
        )

        anchor_after = await request_one(
            session, base_url, anchor, 1, "anchor-after-pressure"
        )

    return {
        "family": spec["family"],
        "model_id": spec["model_id"],
        "apc_enabled": apc_enabled,
        "phase": "eviction_probe",
        "unique_pressure_prefixes": unique_prefixes,
        "prefix_tokens": prefix_tokens,
        "anchor_cold_ttft_ms": anchor_cold["ttft_ms"],
        "anchor_warm_ttft_ms": anchor_warm["ttft_ms"],
        "anchor_after_pressure_ttft_ms": anchor_after["ttft_ms"],
        "warm_speedup_before_pressure": (
            anchor_cold["ttft_ms"] / anchor_warm["ttft_ms"]
        ),
        "retained_speedup_after_pressure": (
            anchor_cold["ttft_ms"] / anchor_after["ttft_ms"]
        ),
        "pressure_batch_ms": pressure_ms,
    }

## Matched APC-off and APC-on runs

Each server is stopped before the next mode starts so cache state and GPU memory do not carry across conditions. Logs are retained when a model fails.

In [ ]:
sweep_rows = []
trace_rows = []
trace_request_rows = []
eviction_rows = []
failure_rows = []
server_rows = []

for model_index, spec in enumerate(MODELS):
    print("\n" + "=" * 96)
    print(spec["family"], spec["model_id"])
    print("=" * 96)

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            spec["model_id"],
            use_fast=True,
        )
    except Exception as error:
        failure_rows.append({
            "family": spec["family"],
            "model_id": spec["model_id"],
            "stage": "tokenizer_load",
            "error": repr(error),
            "traceback": traceback.format_exc(),
        })
        continue

    for apc_enabled in (False, True):
        process = None
        port = find_free_port(PORT_BASE + model_index * 20)
        base_url = f"http://127.0.0.1:{port}"
        started = time.time()

        try:
            process, log_path = launch_server(
                spec=spec,
                apc_enabled=apc_enabled,
                port=port,
            )
            model_info = wait_for_server(
                base_url, process, SERVER_START_TIMEOUT_S
            )
            startup_s = time.time() - started

            server_rows.append({
                "family": spec["family"],
                "model_id": spec["model_id"],
                "apc_enabled": apc_enabled,
                "port": port,
                "startup_s": startup_s,
                "log_path": str(log_path),
                "model_info_json": json.dumps(model_info),
            })

            rows = asyncio.run(run_length_sweep(
                tokenizer=tokenizer,
                base_url=base_url,
                spec=spec,
                apc_enabled=apc_enabled,
            ))
            sweep_rows.extend(rows)

            rows = asyncio.run(run_output_sensitivity(
                tokenizer=tokenizer,
                base_url=base_url,
                spec=spec,
                apc_enabled=apc_enabled,
            ))
            sweep_rows.extend(rows)

            requests, trace_summary = asyncio.run(run_zipf_trace(
                tokenizer=tokenizer,
                base_url=base_url,
                spec=spec,
                apc_enabled=apc_enabled,
            ))
            if trace_summary:
                trace_rows.append(trace_summary)
                for request in requests:
                    trace_request_rows.append({
                        "family": spec["family"],
                        "model_id": spec["model_id"],
                        "apc_enabled": apc_enabled,
                        **request,
                    })

            eviction = asyncio.run(run_eviction_probe(
                tokenizer=tokenizer,
                base_url=base_url,
                spec=spec,
                apc_enabled=apc_enabled,
            ))
            if eviction:
                eviction_rows.append(eviction)

        except Exception as error:
            failure_rows.append({
                "family": spec["family"],
                "model_id": spec["model_id"],
                "apc_enabled": apc_enabled,
                "stage": "server_or_benchmark",
                "error": repr(error),
                "traceback": traceback.format_exc(),
            })
            print("FAILED:", repr(error))
        finally:
            stop_server(process)
            gc.collect()

sweep_df = pd.DataFrame(sweep_rows)
trace_df = pd.DataFrame(trace_rows)
trace_requests_df = pd.DataFrame(trace_request_rows)
eviction_df = pd.DataFrame(eviction_rows)
failures_df = pd.DataFrame(failure_rows)
servers_df = pd.DataFrame(server_rows)

sweep_df.to_csv(RESULTS / "vllm_apc_sweep.csv", index=False)
trace_df.to_csv(RESULTS / "vllm_zipf_trace.csv", index=False)
trace_requests_df.to_csv(
    RESULTS / "vllm_zipf_trace_requests.csv", index=False
)
eviction_df.to_csv(RESULTS / "vllm_eviction_probe.csv", index=False)
failures_df.to_csv(RESULTS / "failures.csv", index=False)
servers_df.to_csv(RESULTS / "server_startup.csv", index=False)

display(sweep_df.head())
display(failures_df)

## Matched comparison and bootstrap confidence intervals

In [ ]:
comparison_rows = []

if not sweep_df.empty:
    key_columns = [
        "family",
        "model_id",
        "phase",
        "workload",
        "target_prefix_tokens",
        "actual_prefix_tokens",
        "concurrency",
        "round",
        "requests",
        "max_tokens",
    ]

    off = sweep_df[sweep_df["apc_enabled"] == False].copy()
    on = sweep_df[sweep_df["apc_enabled"] == True].copy()

    merged = off.merge(
        on,
        on=key_columns,
        suffixes=("_off", "_on"),
        how="inner",
    )

    merged["ttft_speedup"] = (
        merged["median_ttft_ms_off"]
        / merged["median_ttft_ms_on"]
    )
    merged["e2e_speedup"] = (
        merged["median_e2e_ms_off"]
        / merged["median_e2e_ms_on"]
    )
    merged["throughput_speedup"] = (
        merged["requests_per_s_on"]
        / merged["requests_per_s_off"]
    )
    merged["amortized_speedup"] = (
        merged["amortized_total_ms_off"]
        / merged["amortized_total_ms_on"]
    )
    merged.to_csv(
        RESULTS / "vllm_apc_matched_comparison.csv",
        index=False,
    )

    length_only = merged[merged["phase"] == "length_sweep"]
    group_keys = [
        "family",
        "workload",
        "target_prefix_tokens",
        "concurrency",
    ]

    for keys, group in length_only.groupby(group_keys):
        family, workload, context, concurrency = keys

        off_requests = []
        on_requests = []
        for raw in group["request_records_json_off"]:
            off_requests.extend(json.loads(raw))
        for raw in group["request_records_json_on"]:
            on_requests.extend(json.loads(raw))

        off_ttft = [
            row["ttft_ms"]
            for row in off_requests if row["error"] is None
        ]
        on_ttft = [
            row["ttft_ms"]
            for row in on_requests if row["error"] is None
        ]
        estimate, ci_low, ci_high = bootstrap_speedup(
            off_ttft, on_ttft
        )

        comparison_rows.append({
            "family": family,
            "workload": workload,
            "target_prefix_tokens": context,
            "concurrency": concurrency,
            "rows": len(group),
            "ttft_speedup_median": estimate,
            "ttft_speedup_ci95_low": ci_low,
            "ttft_speedup_ci95_high": ci_high,
            "e2e_speedup_median": group["e2e_speedup"].median(),
            "throughput_speedup_median": (
                group["throughput_speedup"].median()
            ),
            "amortized_speedup_median": (
                group["amortized_speedup"].median()
            ),
            "apc_hit_rate_median": (
                group["prefix_cache_hit_rate_on"].median()
            ),
            "all_requests_successful": bool(
                (
                    group["failed_requests_off"]
                    + group["failed_requests_on"]
                ).sum() == 0
            ),
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(
    RESULTS / "vllm_apc_summary.csv", index=False
)
display(comparison_df)

## Evaluation criteria

The benchmark checks request success, long-prefix TTFT speedup, throughput speedup, cache-hit metrics when available, and bootstrap confidence intervals.

In [ ]:
verdict_rows = []

if not comparison_df.empty:
    for family, group in comparison_df.groupby("family"):
        long_group = group[
            group["target_prefix_tokens"] >= 1024
        ].copy()
        if long_group.empty:
            long_group = group.copy()

        finite_hit = long_group[
            np.isfinite(long_group["apc_hit_rate_median"])
        ]
        hit_pass = (
            finite_hit["apc_hit_rate_median"].median() > 0.50
            if not finite_hit.empty
            else True
        )

        ci_positive_rate = float(
            (long_group["ttft_speedup_ci95_low"] > 1.0).mean()
        )

        verdict_rows.append({
            "family": family,
            "conditions": len(long_group),
            "all_requests_successful": bool(
                long_group["all_requests_successful"].all()
            ),
            "median_ttft_speedup_1k_plus": float(
                long_group["ttft_speedup_median"].median()
            ),
            "minimum_ttft_speedup_1k_plus": float(
                long_group["ttft_speedup_median"].min()
            ),
            "median_throughput_speedup_1k_plus": float(
                long_group["throughput_speedup_median"].median()
            ),
            "median_amortized_speedup_1k_plus": float(
                long_group["amortized_speedup_median"].median()
            ),
            "median_apc_hit_rate_1k_plus": (
                float(finite_hit["apc_hit_rate_median"].median())
                if not finite_hit.empty else float("nan")
            ),
            "ci_positive_rate": ci_positive_rate,
            "production_gain_pass": bool(
                long_group["all_requests_successful"].all()
                and long_group["ttft_speedup_median"].median() > 1.20
                and long_group[
                    "throughput_speedup_median"
                ].median() > 1.05
                and hit_pass
                and ci_positive_rate >= 0.50
            ),
        })

verdict_df = pd.DataFrame(verdict_rows)
verdict_df.to_csv(
    RESULTS / "model_production_verdict.csv", index=False
)

requested_families = {model["family"] for model in MODELS}
completed_families = (
    set(verdict_df["family"]) if not verdict_df.empty else set()
)
global_verdict = {
    "profile": PROFILE,
    "requested_families": sorted(requested_families),
    "completed_families": sorted(completed_families),
    "missing_families": sorted(
        requested_families - completed_families
    ),
    "all_models_completed": (
        requested_families == completed_families
    ),
    "all_models_production_gain_pass": (
        bool(verdict_df["production_gain_pass"].all())
        if not verdict_df.empty else False
    ),
    "transformer_exact_reuse_vs_apc_interpretation": (
        "This benchmark measures the existing strong APC baseline. "
        "Parity with APC means the Transformer-compatible MOSAIC path "
        "must not be claimed as novel prefix caching."
    ),
}
global_verdict["global_pass"] = (
    global_verdict["all_models_completed"]
    and global_verdict["all_models_production_gain_pass"]
)

(RESULTS / "global_verdict.json").write_text(
    json.dumps(global_verdict, indent=2),
    encoding="utf-8",
)

display(verdict_df)
print(json.dumps(global_verdict, indent=2))

In [ ]:
if not comparison_df.empty:
    plot_df = (
        comparison_df.groupby(
            ["family", "target_prefix_tokens"],
            as_index=False,
        )
        .agg(
            ttft_speedup=("ttft_speedup_median", "median"),
            e2e_speedup=("e2e_speedup_median", "median"),
            throughput_speedup=(
                "throughput_speedup_median", "median"
            ),
        )
    )

    for family, group in plot_df.groupby("family"):
        plt.plot(
            group["target_prefix_tokens"],
            group["ttft_speedup"],
            marker="o",
            label=family,
        )
    plt.axhline(1.0, linestyle="--")
    plt.xscale("log", base=2)
    plt.xlabel("Exact shared-prefix tokens")
    plt.ylabel("APC OFF TTFT / APC ON TTFT")
    plt.title("vLLM production-style TTFT speedup")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURES / "vllm_apc_ttft_speedup.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    for family, group in plot_df.groupby("family"):
        plt.plot(
            group["target_prefix_tokens"],
            group["throughput_speedup"],
            marker="o",
            label=family,
        )
    plt.axhline(1.0, linestyle="--")
    plt.xscale("log", base=2)
    plt.xlabel("Exact shared-prefix tokens")
    plt.ylabel("APC ON / APC OFF request throughput")
    plt.title("vLLM request-throughput improvement")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURES / "vllm_apc_throughput_speedup.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

if not trace_df.empty:
    trace_plot = trace_df.pivot_table(
        index="family",
        columns="apc_enabled",
        values="requests_per_s",
        aggfunc="median",
    )
    if False in trace_plot.columns and True in trace_plot.columns:
        trace_plot["speedup"] = (
            trace_plot[True] / trace_plot[False]
        )
        trace_plot["speedup"].plot(kind="bar")
        plt.axhline(1.0, linestyle="--")
        plt.ylabel("APC ON / APC OFF throughput")
        plt.title("Zipf repeated-prefix trace")
        plt.tight_layout()
        plt.savefig(
            FIGURES / "zipf_trace_throughput.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.show()

## Outputs

The result archive contains per-request JSON, matched comparisons, bootstrap intervals, server logs, cache metrics, figures, and environment metadata.

In [ ]:
environment = {
    "completed_utc": pd.Timestamp.utcnow().isoformat(),
    "profile": PROFILE,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": GPU_NAME,
    "gpu_memory_bytes": GPU_BYTES,
    "vllm_executable": VLLM_EXE,
    "models": MODELS,
    "config": CFG,
    "server_max_model_len": SERVER_MAX_MODEL_LEN,
    "server_gpu_memory_utilization": (
        SERVER_GPU_MEMORY_UTILIZATION
    ),
    "random_seed": RANDOM_SEED,
}
(RESULTS / "environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

if not verdict_df.empty:
    latex = verdict_df.to_latex(
        index=False,
        float_format=lambda value: f"{value:.3f}",
        caption=(
            "Production-style vLLM automatic-prefix-caching "
            "results by model family."
        ),
        label="tab:vllm_apc_production",
    )
    (RESULTS / "model_production_verdict.tex").write_text(
        latex,
        encoding="utf-8",
    )

readme = f'''# MOSAIC vLLM APC Production Benchmark

Profile: {PROFILE}
GPU: {GPU_NAME}
Requested families: {", ".join(sorted(requested_families))}
Completed families: {", ".join(sorted(completed_families))}
Global pass: {global_verdict["global_pass"]}

This package compares vLLM APC OFF vs ON using streaming TTFT,
end-to-end latency, concurrency, throughput, cache hit metrics,
output-length sensitivity, and a Zipf repeated-prefix trace.

Interpretation:
- APC is the strong exact-prefix baseline.
- A large APC speedup validates the workload opportunity.
- It does not by itself establish novelty over APC.
- MOSAIC novelty must come from broader substring/operator reuse,
  a lower-overhead planner, or an implementation that outperforms
  APC under matched serving conditions.
'''
(ROOT / "README.md").write_text(readme, encoding="utf-8")

zip_path = Path(
    "/kaggle/working/mosaic_vllm_apc_production_results.zip"
)
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for file in ROOT.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(ROOT))

print("Generated:", zip_path)
print("Upload the executed notebook and this ZIP.")